In [195]:
import geopandas as gpd
import pandas as pd
import folium
import osmnx as ox
from shapely.ops import substring
import networkx as nx
from shapely.geometry import Point, LineString, MultiLineString
from multiprocessing import Pool
from functools import partial
from tqdm import tqdm

In [204]:
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')
canopy = gpd.read_file('data/canopy.gdb')
property_full = gpd.read_file('output/hedonic_gdf.gpkg')

pd.set_option('display.max_columns', None)

ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)
# boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('ilam')]

property = gpd.clip(property_full, sa2_chc)
TARGET_CRS = 2193

In [205]:
G = ox.graph_from_place(
  'Christchurch, New Zealand',
  network_type='drive',
  simplify=True,
)

G = ox.project_graph(G, to_crs=TARGET_CRS)

edges = ox.graph_to_gdfs(G, nodes=False)
edges = edges.to_crs(property.crs)

# comptute access point

In [206]:
def nearest_street_and_point(point, edges_gdf, max_distance=50):
    distances = edges_gdf.geometry.distance(point)
    min_distance = distances.min()
    
    if min_distance > max_distance:
        return None
    
    idx = distances.idxmin()
    street_geom = edges_gdf.loc[idx].geometry

    proj_dist = street_geom.project(point)
    access_point = street_geom.interpolate(proj_dist)

    return access_point

property["access_point"] = property.geometry.apply(
    lambda p: nearest_street_and_point(p, edges, max_distance=100)
)

property_valid = property[property['access_point'].notna()].copy()
property = property_valid

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


# iso distance calculation

In [ ]:
def isodistance_from_access_point(property_row, property_crs, G, max_dist=200):

    access_pt = property_row['access_point']
    
    access_pt_proj = gpd.GeoSeries([access_pt], crs=property_crs).to_crs("EPSG:2193").iloc[0]

    # Find nearest edge efficiently using spatial index (front street)
    u, v, k = ox.distance.nearest_edges(G, access_pt_proj.x, access_pt_proj.y, return_dist=False)
    
    data = G.edges[u, v, k]
    
    if 'geometry' in data:
        edge_geom = data['geometry']
    else:
        # Create geometry from node coordinates
        u_pt = Point(G.nodes[u]['x'], G.nodes[u]['y'])
        v_pt = Point(G.nodes[v]['x'], G.nodes[v]['y'])
        edge_geom = LineString([u_pt, v_pt])
        
    edge_length = data['length']
    
    proj_dist = edge_geom.project(access_pt_proj)
    proj_dist = min(max(proj_dist, 0), edge_length)

    dist_to_u = proj_dist
    dist_to_v = edge_length - proj_dist
    
    G_work = G.to_undirected().copy()

    if dist_to_u < 0.5:
        start_node = u
        
    elif dist_to_v < 0.5:
        start_node = v
        
    else:
        access_node = 'access_point'
        G_work.add_node(access_node, x=access_pt_proj.x, y=access_pt_proj.y)

        geom_to_u = substring(edge_geom, 0, proj_dist)
        geom_to_v = substring(edge_geom, proj_dist, edge_length)

        G_work.add_edge(access_node, u, length=dist_to_u, geometry=geom_to_u)
        G_work.add_edge(access_node, v, length=dist_to_v, geometry=geom_to_v)

        if G_work.has_edge(u, v):
            G_work.remove_edge(u, v)
            
        start_node = access_node
    
    lengths = nx.single_source_dijkstra_path_length(
        G_work,
        start_node,
        cutoff=max_dist,
        weight='length'
    )

    lines = []

    for u_node, v_node, data in G_work.edges(data=True):
        
        du = lengths.get(u_node, float('inf'))
        dv = lengths.get(v_node, float('inf'))
        
        if du > max_dist and dv > max_dist:
            continue
                
        geom = data.get('geometry')
        if geom is None:
            u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
            v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
            geom = LineString([u_pt, v_pt])

        edge_len = data.get('length', geom.length)

        if du <= max_dist and dv <= max_dist:
            lines.append(geom)

        elif du <= max_dist < dv:
            remaining = max_dist - du
            if remaining > 0:
                u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
                v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
                
                geom_start = Point(geom.coords[0])
                
                if geom_start.distance(u_pt) < geom_start.distance(v_pt):
                    clipped = substring(geom, 0, min(remaining, edge_len))
                else:
                    start_dist = max(0, edge_len - remaining)
                    clipped = substring(geom, start_dist, edge_len)
                
                if clipped and not clipped.is_empty:
                    lines.append(clipped)

        elif dv <= max_dist < du:
            remaining = max_dist - dv
            if remaining > 0:
                u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
                v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
                
                geom_start = Point(geom.coords[0])
                
                if geom_start.distance(v_pt) < geom_start.distance(u_pt):
                    clipped = substring(geom, 0, min(remaining, edge_len))
                else:
                    start_dist = max(0, edge_len - remaining)
                    clipped = substring(geom, start_dist, edge_len)
                
                if clipped and not clipped.is_empty:
                    lines.append(clipped)

    if not lines:
        return MultiLineString([])

    result = MultiLineString(lines)
    return gpd.GeoSeries([result], crs=G.graph['crs']).to_crs(property_crs).iloc[0]

# for all properties

In [ ]:
def compute_reach_and_canopy(row, property_crs, G, canopy_gdf, max_dist=200, buffer_dist=10):
    try:
        reach = isodistance_from_access_point(
            row,
            property_crs,
            G,
            max_dist=max_dist
        )

        if reach is None or reach.is_empty:
            return MultiLineString([]), None, 0.0

        reach_buffer = reach.buffer(buffer_dist)

        if reach_buffer.is_empty:
            return reach, reach_buffer, 0.0

        buffer_gdf = gpd.GeoDataFrame(
            geometry=[reach_buffer],
            crs=property_crs
        )
        
        canopy_cut = gpd.sjoin(
            canopy_gdf,
            buffer_gdf,
            how='inner',
            predicate='intersects'
        )

        # canopy_cut = gpd.overlay(
        #     canopy_gdf,
        #     buffer_gdf,
        #     how="intersection"
        # )

        canopy_area = canopy_cut.geometry.area.sum()

        return reach, reach_buffer, canopy_area

    except Exception as e:
        print(f"Error for property {row.name}: {e}")
        return MultiLineString([]), None, 0.0

In [ ]:
tqdm.pandas(desc="Computing isodistance")

sub_property = property.sample(n=1000, random_state=2)

results = sub_property.progress_apply(
    lambda row: compute_reach_and_canopy(
        row,
        property.crs,
        G,
        canopy,
        max_dist=200,
        buffer_dist=10
    ),
    axis=1
)

sub_property['reach_200m'] = results.apply(lambda x: x[0])
sub_property['reach_buffer'] = results.apply(lambda x: x[1]) 
sub_property['canopy_area_buffer'] = results.apply(lambda x: x[2])

# plotting

In [210]:
canopy_wgs = canopy.to_crs(4326)
props_wgs = sub_property.geometry.to_crs(4326)

m = folium.Map(
    location=[props_wgs.y.mean(), props_wgs.x.mean()],
    zoom_start=15,
    tiles="CartoDB positron"
)

for idx, row in sub_property.iterrows():

    # ---- reproject core geometries ----
    prop = gpd.GeoSeries([row.geometry], crs=property.crs).to_crs(4326).iloc[0]
    access = gpd.GeoSeries([row['access_point']], crs=property.crs).to_crs(4326).iloc[0]
    reach = gpd.GeoSeries([row['reach_200m']], crs=property.crs).to_crs(4326).iloc[0]
    buffer_geom = gpd.GeoSeries([row['reach_buffer']], crs=property.crs).to_crs(4326).iloc[0]

    # ---- property ----
    folium.CircleMarker(
        [prop.y, prop.x],
        radius=7,
        color="purple",
        fill=True,
        fill_opacity=1,
        popup=f"Property {idx}"
    ).add_to(m)

    # ---- access point ----
    folium.CircleMarker(
        [access.y, access.x],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.9,
        popup=f"Access {idx}"
    ).add_to(m)

    # ---- reachable streets ----
    if not reach.is_empty:
        folium.GeoJson(
            reach,
            style_function=lambda x: {
                "color": "red",
                "weight": 3,
                "opacity": 0.6
            },
            tooltip=f"Reachable streets ({idx})"
        ).add_to(m)

    # ---- street buffer ----
    if buffer_geom is not None and (not buffer_geom.is_empty):
        folium.GeoJson(
            buffer_geom,
            style_function=lambda x: {
                "color": "orange",
                "fillColor": "orange",
                "weight": 1,
                "fillOpacity": 0.25
            },
            tooltip=f"5m street buffer ({idx})"
        ).add_to(m)

    # ---- canopy intersecting buffer ----
    if buffer_geom is not None and (not buffer_geom.is_empty):
        canopy_clip = canopy_wgs[canopy_wgs.intersects(buffer_geom)]

        if not canopy_clip.empty:
            folium.GeoJson(
                canopy_clip,
                style_function=lambda x: {
                    "color": "darkgreen",
                    "fillColor": "darkgreen",
                    "weight": 1,
                    "fillOpacity": 0.6
                },
                tooltip=f"Canopy in buffer ({idx})"
            ).add_to(m)

m